In [ ]:

#%pip install -q requests beautifulsoup4 nltk unidecode scikit-learn matplotlib seaborn plotly wordcloud joblib pandas

In [1]:
import json
import pathlib
import random
import re
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import requests
import seaborn as sns
from bs4 import BeautifulSoup
from joblib import Parallel, delayed
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from unidecode import unidecode

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
CORES = sns.color_palette("muted")

print("Imports OK")

Imports OK


## 1. Configurações

In [2]:
BASE_URL = "https://motor1.uol.com.br"
NEWS_URL = f"{BASE_URL}/news/"

candidatos_data_dir = [
    pathlib.Path("noticias_motor1"),
    pathlib.Path("Lab/WebScrapping/noticias_motor1"),
]
DATA_DIR = next((p for p in candidatos_data_dir if p.exists()), candidatos_data_dir[0])
DATA_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/117.0.0.0 Safari/537.36"
    )
}

NUMERO_PAGINAS = 40
RODAR_SCRAPING = True

## 2. Coleta de links

In [3]:
def coletar_links_pagina(pagina: int) -> list[str]:
    if pagina == 1:
        url = NEWS_URL
    else:
        url = f"{NEWS_URL}?p={pagina}"

    response = requests.get(url, headers=HEADERS, timeout=15)
    print(f"Página {pagina}: {response.status_code}")
    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    links = []

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
        except (TypeError, json.JSONDecodeError):
            continue

        if data.get("@type") != "ItemList":
            continue

        for item in data.get("itemListElement", []):
            url_item = item.get("url", "")
            if "/news/" in url_item and url_item not in links:
                links.append(url_item)

    if not links:
        for article in soup.find_all("article"):
            a = article.find("a", href=True)
            if not a:
                continue

            href = a["href"]
            if href.startswith("/news/"):
                href = BASE_URL + href

            if href.startswith("http") and "/news/" in href and href not in links:
                links.append(href)

    return links

In [ ]:
if RODAR_SCRAPING:
    todos_os_links = []

    for pagina in range(1, NUMERO_PAGINAS + 1):
        links = coletar_links_pagina(pagina)
        todos_os_links.extend(links)
        time.sleep(random.uniform(0.8, 1.5))

    todos_os_links = list(dict.fromkeys(todos_os_links))
    print(f"Total de links únicos: {len(todos_os_links)}")
    print(todos_os_links[:5])
else:
    todos_os_links = []
    print("Scraping desligado. Altere RODAR_SCRAPING para True se quiser coletar links novos.")

Página 1: 200
Página 2: 200
Página 3: 200
Página 4: 200
Página 5: 200
Página 6: 200
Página 7: 200
Página 8: 200
Página 9: 200
Página 10: 200
Página 11: 410
Página 12: 410
Página 13: 410
Página 14: 410
Página 15: 410
Página 16: 410
Página 17: 410
Página 18: 410
Página 19: 410
Página 20: 410
Página 21: 410
Página 22: 410
Página 23: 410
Página 24: 410
Página 25: 410
Página 26: 410
Página 27: 410
Página 28: 410
Página 29: 410
Página 30: 410
Página 31: 410
Página 32: 410
Página 33: 410
Página 34: 410
Página 35: 410
Página 36: 410
Página 37: 410
Página 38: 410
Página 39: 410
Página 40: 410
Total de links únicos: 199
['https://motor1.uol.com.br/news/796498/vendas-reino-unido-carros-abril/', 'https://motor1.uol.com.br/news/796725/citroen-2cv-retorno-eletrico-barato/', 'https://motor1.uol.com.br/news/796489/vendas-carros-abril-novos-it%C3%A1lia/', 'https://motor1.uol.com.br/news/796750/analise-guerra-potencia-eletricos-chineses/', 'https://motor1.uol.com.br/news/796764/citroen-promocao-basalt-c

## 3. Extração das notícias

In [17]:
def extrair_noticia(url: str) -> dict | None:
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"Erro {response.status_code}: {url}")
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        tag_titulo = soup.find("meta", property="og:title")
        titulo = tag_titulo["content"] if tag_titulo else ""
        if not titulo:
            h1 = soup.find("h1")
            titulo = h1.get_text(strip=True) if h1 else ""

        tag_desc = soup.find("meta", attrs={"name": "description"})
        descricao = tag_desc["content"] if tag_desc else ""

        tag_cat = soup.find("a", class_=lambda c: c and "text-text-contrast-accent" in c)
        categoria = tag_cat.get_text(strip=True) if tag_cat else ""

        tag_time = soup.find("time", attrs={"data-time": True})
        if tag_time:
            data = pd.to_datetime(int(tag_time["data-time"]), unit="s", utc=True).isoformat()
        else:
            tag_dt = soup.find("meta", property="article:published_time")
            data = tag_dt["content"] if tag_dt else ""

        corpo = soup.find("div", class_=lambda c: c and "article" in c.lower())
        corpo = corpo or soup.find("main") or soup.find("section")
        paragrafos = corpo.find_all("p") if corpo else soup.find_all("p")
        texto = "\n".join(
            p.get_text(strip=True)
            for p in paragrafos
            if len(p.get_text(strip=True)) > 30
        )

        return {
            "url": url,
            "titulo": titulo,
            "descricao": descricao,
            "categoria": categoria,
            "data": data,
            "texto": texto,
        }
    except Exception as exc:
        print(f"Falha em {url}: {exc}")
        return None

In [18]:
def salvar_noticia(i: int, url: str) -> None:
    destino = DATA_DIR / f"noticia_{i:04d}.json"
    if destino.exists():
        return

    noticia = extrair_noticia(url)
    if not noticia:
        return

    with destino.open("w", encoding="utf-8") as f:
        json.dump(noticia, f, ensure_ascii=False, indent=2)

    print(f"Salvo: {destino.name} — {noticia['titulo'][:70]}")
    time.sleep(random.uniform(0.2, 0.6))


if RODAR_SCRAPING and todos_os_links:
    Parallel(n_jobs=4, prefer="threads")(
        delayed(salvar_noticia)(i, url)
        for i, url in enumerate(todos_os_links)
    )
    print(f"Coleta concluída em: {DATA_DIR.resolve()}")
else:
    print("Extração pulada. Usando os JSONs já existentes para a análise.")

Salvo: noticia_0195.json — GWM Tank 300 será o primeiro híbrido flex da marca no Brasil
Salvo: noticia_0194.json — MG4 Urban chega em junho e pode ser produzido no Brasil em 2026
Salvo: noticia_0193.json — Segredo: novo GWM Ora 5 será feito no Brasil com versões flex e híbrid
Salvo: noticia_0192.json — Estreia mundial: Novo Porsche Cayenne Coupé Electric une luxo e 1.156 
Salvo: noticia_0196.json — Yamaha lança ZR Hybrid Connected 2026 no Brasil por R$ 13.990
Salvo: noticia_0197.json — BYD confirma Dolphin híbrido em 2026 e mantém King no Brasil
Salvo: noticia_0198.json — Com Omoda 4 e Jaecoo 5, marca da Chery terá SUVs ainda mais baratos em
Coleta concluída em: C:\Users\muvil\Nova pasta\analise_site_motor1_uol\noticias_motor1
